[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/Latent/blob/main/notebooks/colab/Appendix_1_Sentiment_Benchmark.ipynb)

---


In [ ]:
# ══════════════════════════════════════════
# Google Colab Setup
# ══════════════════════════════════════════
# Run this cell first to set up the environment in Google Colab.

import os
import sys

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Google Drive for data persistence
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set project root on Google Drive
    PROJECT_ROOT = '/content/drive/MyDrive/Latent'
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/intermediate_data', exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/utilities', exist_ok=True)
    
    # Clone or update the repo
    if not os.path.exists('/content/Latent'):
        !git clone https://github.com/YOUR_USERNAME/Latent.git /content/Latent
    
    # Install dependencies
    !pip install -q spacy nltk ftfy emoji langdetect sentence-transformers rank-bm25 tqdm
    !python -m spacy download en_core_web_sm
    
    # Set working directory
    os.chdir('/content/Latent')
    print(f'Working directory: {os.getcwd()}')
    print(f'Data directory: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = '.'
    print('Running locally. Ensure dependencies are installed via: pip install -r requirements.txt')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Define the path to your CSV file
file_path = '/content/drive/MyDrive/Latent/data/intermediate_data/stopword_lemmatized_posts_0.csv'

# Read the CSV file into a pandas DataFrame
df_sample = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
display(df_sample.head())

In [ ]:
# # Select 50 random samples from df_sample
# random_samples = df_sample.sample(n=50, random_state=42) # Using random_state for reproducibility

# # Display the 'post_id' and 'cleaned_text_normalized' columns
# display(random_samples[['id', 'cleaned_text_normalized']])

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/Latent/data/posts_sampled.csv'
df = pd.read_csv(file_path, usecols=['id', 'cleaned_text_normalized'])
display(df.head())

First, I'll install the `transformers` library, which is necessary for using Hugging Face models. The `[sentencepiece]` extra is often needed for some tokenizer models.

In [ ]:
!pip install transformers[sentencepiece] -q

In [ ]:
import huggingface_hub
from huggingface_hub import login

your_huggingface_token = "..." # Replace with your token
login(token=your_huggingface_token)

Next, I'll load both specified emotion prediction models and their respective tokenizers using the `pipeline` function from the `transformers` library. I'll then process the first 100 rows of your DataFrame to predict emotions using each model.

In [ ]:
from transformers import pipeline

# Model 1: bert-base-go-emotion
model_name_1 = "bhadresh-savani/bert-base-go-emotion" #28 labels
print(f"Loading Model 1: {model_name_1}")
classifier_1 = pipeline("text-classification", model=model_name_1, return_all_scores=True)

# Model 2: emotion-english-distilroberta-base
model_name_2 = "j-hartmann/emotion-english-distilroberta-base" # 7 labels
print(f"Loading Model 2: {model_name_2}")
classifier_2 = pipeline("text-classification", model=model_name_2, return_all_scores=True)

# Model 3: emotion-english-roberta-large
model_name_3 = "j-hartmann/emotion-english-roberta-large" # 7 labels
print(f"Loading Model 3: {model_name_3}")
classifier_3 = pipeline("text-classification", model=model_name_3, return_all_scores=True)

# Model 4: roberta-base-go_emotions
model_name_4 = "SamLowe/roberta-base-go_emotions" #28
print(f"Loading Model 4: {model_name_4}")
classifier_4 = pipeline("text-classification", model=model_name_4, return_all_scores=True)

# Model 5: bert-base-cased-goemotions-original
model_name_5 = "monologg/bert-base-cased-goemotions-original" #28
classifier_5 = pipeline("text-classification", model=model_name_5, return_all_scores=True)

# Select the first 50 rows of the DataFrame
df_subset = df.head(50).copy()

# Helper function to extract the dominant emotion label and its score, and all predictions
def get_dominant_emotion_score_and_all(predictions):
    if not predictions:
        return None, None, []
    dominant_prediction = max(predictions, key=lambda x: x['score'])
    return dominant_prediction['label'], dominant_prediction['score'], predictions

text_column_name = "cleaned_text_normalized"

# Store results
emotions_model_1 = []
scores_model_1 = []
all_predictions_model_1 = []
emotions_model_2 = []
scores_model_2 = []
all_predictions_model_2 = []
emotions_model_3 = []
scores_model_3 = []
all_predictions_model_3 = []
emotions_model_4 = []
scores_model_4 = []
all_predictions_model_4 = []
emotions_model_5 = []
scores_model_5 = []
all_predictions_model_5 = []

for index, row in df_subset.iterrows():
    text = str(row[text_column_name])

    # Predict with Model 1
    pred_1 = classifier_1(text)
    emotion_1, score_1, all_1 = get_dominant_emotion_score_and_all(pred_1)
    emotions_model_1.append(emotion_1)
    scores_model_1.append(score_1)
    all_predictions_model_1.append(all_1)

    # Predict with Model 2
    pred_2 = classifier_2(text)
    emotion_2, score_2, all_2 = get_dominant_emotion_score_and_all(pred_2)
    emotions_model_2.append(emotion_2)
    scores_model_2.append(score_2)
    all_predictions_model_2.append(all_2)

    # Predict with Model 3
    pred_3 = classifier_3(text)
    emotion_3, score_3, all_3 = get_dominant_emotion_score_and_all(pred_3)
    emotions_model_3.append(emotion_3)
    scores_model_3.append(score_3)
    all_predictions_model_3.append(all_3)

    # Predict with Model 4
    pred_4 = classifier_4(text)
    emotion_4, score_4, all_4 = get_dominant_emotion_score_and_all(pred_4)
    emotions_model_4.append(emotion_4)
    scores_model_4.append(score_4)
    all_predictions_model_4.append(all_4)

    # Predict with Model 5
    pred_5 = classifier_5(text)
    emotion_5, score_5, all_5 = get_dominant_emotion_score_and_all(pred_5)
    emotions_model_5.append(emotion_5)
    scores_model_5.append(score_5)
    all_predictions_model_5.append(all_5)

df_subset[f'emotion_{model_name_1}'] = emotions_model_1
df_subset[f'score_{model_name_1}'] = scores_model_1
df_subset[f'all_predictions_{model_name_1}'] = all_predictions_model_1
df_subset[f'emotion_{model_name_2}'] = emotions_model_2
df_subset[f'score_{model_name_2}'] = scores_model_2
df_subset[f'all_predictions_{model_name_2}'] = all_predictions_model_2
df_subset[f'emotion_{model_name_3}'] = emotions_model_3
df_subset[f'score_{model_name_3}'] = scores_model_3
df_subset[f'all_predictions_{model_name_3}'] = all_predictions_model_3
df_subset[f'emotion_{model_name_4}'] = emotions_model_4
df_subset[f'score_{model_name_4}'] = scores_model_4
df_subset[f'all_predictions_{model_name_4}'] = all_predictions_model_4
df_subset[f'emotion_{model_name_5}'] = emotions_model_5
df_subset[f'score_{model_name_5}'] = scores_model_5
df_subset[f'all_predictions_{model_name_5}'] = all_predictions_model_5

In [ ]:
display(df_subset.head())


In [ ]:
output_file_path = '/content/drive/MyDrive/Latent/data/sampled_labels.csv'
df_subset.to_csv(output_file_path, index=False)
print(f"DataFrame saved to: {output_file_path}")

### Processing `stopword_lemmatized_comments_1_onlytext.csv`

Now, I will load the new comments file, apply the same emotion prediction models to its first 50 rows, and store the results in a new DataFrame.

In [ ]:
import pandas as pd

comments_file_path = '/content/drive/MyDrive/Latent/data/intermediate_data/stopword_lemmatized_comments_1_onlytext.csv'
df_comments = pd.read_csv(comments_file_path)
df_comments_subset = df_comments.head(50).copy()

print(f"Loaded {comments_file_path} and selected the first 50 rows.")
display(df_comments_subset.head())

In [ ]:
# Store results for comments dataframe
emotions_model_1_c = []
scores_model_1_c = []
emotions_model_2_c = []
scores_model_2_c = []
emotions_model_3_c = []
scores_model_3_c = []
emotions_model_4_c = []
scores_model_4_c = []
emotions_model_5_c = []
scores_model_5_c = []

for index, row in df_comments_subset.iterrows():
    text = str(row[text_column_name])

    # Predict with Model 1
    pred_1 = classifier_1(text)
    emotion_1, score_1 = get_dominant_emotion_and_score(pred_1)
    emotions_model_1_c.append(emotion_1)
    scores_model_1_c.append(score_1)

    # Predict with Model 2
    pred_2 = classifier_2(text)
    emotion_2, score_2 = get_dominant_emotion_and_score(pred_2)
    emotions_model_2_c.append(emotion_2)
    scores_model_2_c.append(score_2)

    # Predict with Model 3
    pred_3 = classifier_3(text)
    emotion_3, score_3 = get_dominant_emotion_and_score(pred_3)
    emotions_model_3_c.append(emotion_3)
    scores_model_3_c.append(score_3)

    # Predict with Model 4
    pred_4 = classifier_4(text)
    emotion_4, score_4 = get_dominant_emotion_and_score(pred_4)
    emotions_model_4_c.append(emotion_4)
    scores_model_4_c.append(score_4)

    # Predict with Model 5
    pred_5 = classifier_5(text)
    emotion_5, score_5 = get_dominant_emotion_and_score(pred_5)
    emotions_model_5_c.append(emotion_5)
    scores_model_5_c.append(score_5)

df_comments_subset[f'emotion_{model_name_1}'] = emotions_model_1_c
df_comments_subset[f'score_{model_name_1}'] = scores_model_1_c
df_comments_subset[f'emotion_{model_name_2}'] = emotions_model_2_c
df_comments_subset[f'score_{model_name_2}'] = scores_model_2_c
df_comments_subset[f'emotion_{model_name_3}'] = emotions_model_3_c
df_comments_subset[f'score_{model_name_3}'] = scores_model_3_c
df_comments_subset[f'emotion_{model_name_4}'] = emotions_model_4_c
df_comments_subset[f'score_{model_name_4}'] = scores_model_4_c
df_comments_subset[f'emotion_{model_name_5}'] = emotions_model_5_c
df_comments_subset[f'score_{model_name_5}'] = scores_model_5_c

### Displaying and Saving Results for Comments

Here are the first few rows of the comments DataFrame with the emotion predictions, and then I will save it to a new CSV file.

In [ ]:
display(df_comments_subset.head())

In [ ]:
output_file_path_comments = '/content/drive/MyDrive/Latent/data/xinyliu_test_label_comments.csv'
df_comments_subset.to_csv(output_file_path_comments, index=False)
print(f"Comments DataFrame saved to: {output_file_path_comments}")

In [ ]:
output_file_path_sampled = '/content/drive/MyDrive/Latent/data/posts_sampled.csv'
random_samples.to_csv(output_file_path_sampled, index=False)
print(f"Sampled posts DataFrame saved to: {output_file_path_sampled}")